# BACE-XGNN: GPU D-MPNN out-of-fold run

**Before running:** (1) upload the `bace-xgnn` folder as a Kaggle Dataset called `bace-xgnn` (it must contain `backend/` and `data/bace.csv`), (2) add it to this notebook via *Add Input*, (3) Settings > Accelerator > **GPU T4 x2** or P100, (4) Internet **On** (needed to pip install RDKit).

Output: `/kaggle/working/gnn_oof.npz`. Download it and put it in `bace-xgnn/` locally.

In [ ]:
!pip -q install rdkit torch_geometric
import torch; print(torch.__version__, 'cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on a GPU accelerator in Settings'

In [ ]:
import glob, os, shutil
src = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/backend/bace/gnn.py', recursive=True)[0]))
src = os.path.dirname(src)
dst = '/kaggle/working/bace-xgnn'
if os.path.exists(dst): shutil.rmtree(dst)
shutil.copytree(src, dst)
os.chdir(dst + '/backend')
print(os.getcwd(), os.listdir('..'))
assert os.path.exists('../data/bace.csv')

## Smoke test (1 split, 1 seed). Should take a minute or two. Check the speed before the full run.

In [ ]:
!python -m bace.cloud_run --out /kaggle/working/smoke.npz --seeds 1 --only random0

## Full run: 9 outer splits x (5 inner + 1 final) fits x 3 seeds. Saved after every split.

In [ ]:
!python -m bace.cloud_run --out /kaggle/working/gnn_oof.npz --seeds 3

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import sys; sys.path.insert(0, '.')
from bace import data
df = data.load(); y = df['y'].values
z = np.load('/kaggle/working/gnn_oof.npz')
for k in sorted({n.split('__')[0] for n in z.files}):
    te = z[k+'__te']; p = z[k+'__test']
    print(f'{k:10s} GNN-only acc@0.5={accuracy_score(y[te], p>.5):.4f} auc={roc_auc_score(y[te], p):.4f}')